Eval Trace Inspector\n\nParse a LegalBench-RAG eval `.jsonl` trace, display per-query metrics, and cross-check hit chunks against OpenSearch and the raw corpus."

## Config

In [1]:
TRACE_FILE   = "logs/eval/test_set.jsonl"   # path to the .jsonl trace
CORPUS_DIR   = "data/test_set/corpus"       # root of the corpus used in this run
OS_URL       = "http://localhost:9200"
INDEX_NAME   = "test_set"                      # OpenSearch index name used during eval

## Parse trace

In [2]:
import json

def load_trace(path):
    records, summary = [], None
    for line in open(path, encoding="utf-8"):
        line = line.strip()
        if not line.startswith("{"):
            continue
        obj = json.loads(line)
        if obj.get("_type") == "run_summary":
            summary = obj
        else:
            records.append(obj)
    return records, summary

query_records, summary = load_trace(TRACE_FILE)

print(f"Loaded {len(query_records)} query records")
print(f"Run info: {summary['run_info']}")

Loaded 3 query records
Run info: {'label': 'sentence-transformers/all-mpnet-base-v2', 'embedding_model': 'sentence-transformers/all-mpnet-base-v2', 'index_name': 'test_set', 'ks': [2, 4, 6, 10, 15, 20, 40], 'timestamp': '2026-04-23T11:41:42'}


## Aggregate metrics table"

In [5]:
import pandas as pd

rows = []
for bname, bmetrics in summary["benchmarks"].items():
    for k_str, val in bmetrics["chunk_recall_at_k"].items():
        rows.append({
            "benchmark": bname,
            "K": int(k_str),
            "chunk_recall": val,
            "chunk_precision": bmetrics["chunk_precision_at_k"][k_str],
            "char_recall": bmetrics["char_recall_at_k"][k_str],
            "char_precision": bmetrics["char_precision_at_k"][k_str],
        })

df = pd.DataFrame(rows).set_index(["benchmark", "K"])
df

chunk_recall  chunk_precision  char_recall  char_precision
benchmark K                                                             
cuad      2       0.000000         0.000000     0.000000        0.000000
          4       0.000000         0.000000     0.000000        0.000000
          6       0.333333         0.055556     0.333333        0.017176
          10      0.333333         0.033333     0.333333        0.010219
          15      0.333333         0.022222     0.333333        0.007099
          20      1.000000         0.050000     1.000000        0.031944
          40      1.000000         0.025000     1.000000        0.016057

## Per-query metrics"

In [10]:
for r in query_records:
    print(f"Q{r['query_idx']}: {r['query'][:90]}...")
    for gt in r["ground_truth"]:
        print(f"  GT  file={gt['file']}  span={gt['span']}")
    print(f"  retrieved={r['n_retrieved']}  total_gt_chars={r['total_gt_chars']}")
    header = f"  {'K':}  {'chunk_rec':}  {'char_rec':}  {'gt_hit':}"
    print(header)
    for m in r["metrics_by_k"]:
        print(f"  {m['k']:}  {m['chunk_recall']:}  {m['char_recall']:}  {m['n_gt_hit']:}/{m['n_gt_snippets']}")
    print()

Q1: Consider the Marketing Affiliate Agreement between Birch First Global Investments Inc. and...
  GT  file=cuad/CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.txt  span=[1414, 1784]
  retrieved=40  total_gt_chars=370
  K  chunk_rec  char_rec  gt_hit
  2  0.0  0.0  0/1
  4  0.0  0.0  0/1
  6  0.0  0.0  0/1
  10  0.0  0.0  0/1
  15  0.0  0.0  0/1
  20  1.0  1.0  1/1
  40  1.0  1.0  1/1

Q2: Consider the Marketing Affiliate Agreement between Birch First Global Investments Inc. and...
  GT  file=cuad/CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.txt  span=[1414, 1784]
  retrieved=40  total_gt_chars=370
  K  chunk_rec  char_rec  gt_hit
  2  0.0  0.0  0/1
  4  0.0  0.0  0/1
  6  0.0  0.0  0/1
  10  0.0  0.0  0/1
  15  0.0  0.0  0/1
  20  1.0  1.0  1/1
  40  1.0  1.0  1/1

Q3: Consider the Marketing Affiliate Agreement between Birch First Global Investments Inc. and...
  GT  file=cuad/CybergyHoldingsInc_20140520_10-Q_EX-10.27_

check hit chunks for a specific query

Set `QUERY_IDX` and `AT_K` to drill into any query."

In [31]:
QUERY_IDX = 3   # 1-based, matches query_idx in the trace
AT_K      = 20  # which K cutoff to inspect

r = next(x for x in query_records if x["query_idx"] == QUERY_IDX)
m = next(x for x in r["metrics_by_k"] if x["k"] == AT_K)

print(f"Query: {r['query']}")
print(f"GT spans: {[g['span'] for g in r['ground_truth']]}")
print(f"\nHit chunks at K={AT_K}:")
for c in m["chunk_hits"]:
    tag = "HIT " if c["is_chunk_hit"] else "miss"
    print(f"  [{tag}] rank={c['rank']:>3}  chars={c['char_start']}-{c['char_end']}  score={c['score']:.6f}  id={c['chunk_id']}")
    for ov in c.get("gt_overlaps", []):
        print(f"         overlap={ov['overlap_span']}  ({ov['overlap_chars']} chars, {ov['overlap_pct_of_gt']:.1f}% of GT)")

Query: Consider the Marketing Affiliate Agreement between Birch First Global Investments Inc. and Mount Knowledge Holdings Inc.; What is the notice period required to terminate the renewal?
GT spans: [[22066, 22221]]

Hit chunks at K=20:
  [miss] rank=  1  chars=0-512  score=0.032787  id=d98a438e4ece918856ae260dbac33982
  [miss] rank=  2  chars=31126-31638  score=0.030777  id=48a84d204b2f6980a748e2cc0596a215
  [miss] rank=  3  chars=448-960  score=0.030622  id=df72bbe95c0332bbcbd5817fe74d1a2d
  [miss] rank=  4  chars=30376-30888  score=0.029710  id=182cb8714eb3fb71235e49a4aac7aa45
  [miss] rank=  5  chars=16646-17158  score=0.028595  id=680228eacff2ab60655796f00c703d6b
  [HIT ] rank=  6  chars=21839-22351  score=0.028439  id=180d29d5c1740ff760ee1a894ae54102
         overlap=[22066, 22221]  (155 chars, 100.0% of GT)
  [miss] rank=  7  chars=15302-15814  score=0.027921  id=d39fdc8c4f75c65bf83fd2629ca32ae3
  [miss] rank=  8  chars=27871-28383  score=0.027730  id=58ceefe69bf3cae204874f7546

## Fetch a chunk from OpenSearch and verify GT overlap"

In [ ]:
import urllib.request, pathlib

def fetch_chunk(chunk_id, index=INDEX_NAME, base_url=OS_URL):
    url = f"{base_url}/{index}/_doc/{chunk_id}"
    with urllib.request.urlopen(url) as resp:
        return json.loads(resp.read())["_source"]

def corpus_text(file_path, corpus_dir=CORPUS_DIR):
    return pathlib.Path(corpus_dir, file_path).read_text(encoding="utf-8")

# --- pick any chunk_id from the cell above ---
CHUNK_ID = "180d29d5c1740ff760ee1a894ae54102"

chunk = fetch_chunk(CHUNK_ID)
print(f"chunk_id  : {chunk['chunk_id']}")
print(f"citation  : {chunk['citation']}")
print(f"char range: {chunk['char_start']} – {chunk['char_end']}")
print(f"\n--- chunk text ---\n{chunk['text']}")

chunk_id  : 180d29d5c1740ff760ee1a894ae54102
citation  : cuad/CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.txt
char range: 21839 – 22351
is_parent : False

--- chunk text ---
ely responsible for any claims, warranties or representations made by MA or MA's representatives or agents, which differ from the warranties, provided by Company in the applicable end user license agreement(s).   Termination.

This Agreement may be terminated by either party at the expiration of its term or any renewal term upon thirty (30) days written notice to the other party. Company acknowledges that this Agreement shall not be terminated for MA's failure to follow an operating plan, standard procedure


In [ ]:
# --- pick any chunk_id from the cell above ---
CHUNK_ID = "b9c2075d4a888374503c939cd153c0ac"

chunk = fetch_chunk(CHUNK_ID)
print(f"chunk_id  : {chunk['chunk_id']}")
print(f"citation  : {chunk['citation']}")
print(f"char range: {chunk['char_start']} – {chunk['char_end']}")
print(f"\n--- chunk text ---\n{chunk['text']}")

chunk_id  : b9c2075d4a888374503c939cd153c0ac
citation  : cuad/2ThemartComInc_19990826_10-12G_EX-10.10_6700288_EX-10.10_Co-Branding Agreement_ Agency Agreement.txt
char range: 0 – 512
is_parent : False

--- chunk text ---
CO-BRANDING AND ADVERTISING AGREEMENT

THIS CO-BRANDING AND ADVERTISING AGREEMENT (the "Agreement") is made as of June 21, 1999 (the "Effective Date") by and between I-ESCROW, INC., with its principal place of business at 1730 S. Amphlett Blvd., Suite 233, San Mateo, California 94402 ("i-Escrow"), and 2THEMART.COM, INC. having its principal place of business at 18301 Von Karman Avenue, 7th Floor, Irvine, California 92612 ("2TheMart").

1. DEFINITIONS.

(a) "CONTENT" means all content or information, in any 


In [33]:
hit_chuck = fetch_chunk("180d29d5c1740ff760ee1a894ae54102")
print(f"chunk_id  : {hit_chuck['chunk_id']}")
print(f"citation  : {hit_chuck['citation']}")
print(f"char range: {hit_chuck['char_start']} – {hit_chuck['char_end']}")
print(f"\n--- chunk text ---\n{hit_chuck['text']}")

chunk_id  : 180d29d5c1740ff760ee1a894ae54102
citation  : cuad/CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.txt
char range: 21839 – 22351

--- chunk text ---
ely responsible for any claims, warranties or representations made by MA or MA's representatives or agents, which differ from the warranties, provided by Company in the applicable end user license agreement(s).   Termination.

This Agreement may be terminated by either party at the expiration of its term or any renewal term upon thirty (30) days written notice to the other party. Company acknowledges that this Agreement shall not be terminated for MA's failure to follow an operating plan, standard procedure


In [34]:
# Verify: slice GT span from raw corpus and check it's inside the chunk text
gt_span = r["ground_truth"][0]["span"]   # [start, end]

doc_text = corpus_text(hit_chuck["citation"])
gt_text    = doc_text[gt_span[0] : gt_span[1]]
chunk_text = doc_text[hit_chuck["char_start"] : hit_chuck["char_end"]]

print(f"raw corpus chunk text (not in opensearch):\n{chunk_text}")
print("-------------------------------")

print(f"opensearch chuck text:\n{hit_chuck['text']}")
print("-------------------------------")

print(f"GT span [{gt_span[0]}:{gt_span[1]}]:")
print(f"gt text:{gt_text}")
print("-------------------------------")

print(f"GT fully inside corpus slice of chunk? : {gt_text in chunk_text}")
print(f"GT fully inside OpenSearch chunk text? : {gt_text in hit_chuck['text']}")

raw corpus chunk text (not in opensearch):
ely responsible for any claims, warranties or representations made by MA or MA's representatives or agents, which differ from the warranties, provided by Company in the applicable end user license agreement(s).   Termination.

This Agreement may be terminated by either party at the expiration of its term or any renewal term upon thirty (30) days written notice to the other party. Company acknowledges that this Agreement shall not be terminated for MA's failure to follow an operating plan, standard procedure
-------------------------------
opensearch chuck text:
ely responsible for any claims, warranties or representations made by MA or MA's representatives or agents, which differ from the warranties, provided by Company in the applicable end user license agreement(s).   Termination.

This Agreement may be terminated by either party at the expiration of its term or any renewal term upon thirty (30) days written notice to the other party. Compan